# Data Sampling and Preparation

This module works through Chapter 2 which covers how text data is prepared as LLM input. 

## Tokenizing Text

This is the first step in preparing text data. We need to go from input text -> tokenized text -> token IDs -> token embeddings. First, we download the short story "The Verdict" by Edith Wharton which has been released to the public domain and is thus permitted for LLM training tasks. This story is what we will tokenize.

In [14]:
import urllib.request

url = ("https://raw.githubusercontent.com/rasbt/"
       "LLMs-from-scratch/main/ch02/01_main-chapter-code/"
       "the-verdict.txt")
file_path = "the-verdict.txt"
urllib.request.urlretrieve(url, file_path)

('the-verdict.txt', <http.client.HTTPMessage at 0x2554bbe3130>)

In [15]:
with open("the-verdict.txt", "r", encoding="utf-8") as f:
    raw_text = f.read()
print(f"Total number of characters is: {len(raw_text):,}")
print(raw_text[:99])

Total number of characters is: 20,479
I HAD always thought Jack Gisburn rather a cheap genius--though a good fellow enough--so it was no 


A most simple tokenizer might split by whitespace.

In [16]:
# Simple example tokenizer
import re
text = "Hello world. This, is a test."
result = re.split(r'(\s)', text)
print(result)

['Hello', ' ', 'world.', ' ', 'This,', ' ', 'is', ' ', 'a', ' ', 'test.']


In [17]:
# Let's separate out punctuation characters too
result = re.split(r'([,.]|\s)', text)
print(result)

['Hello', ' ', 'world', '.', '', ' ', 'This', ',', '', ' ', 'is', ' ', 'a', ' ', 'test', '.', '']


In [18]:
# Remove redundant whitespace characters
result = [item for item in result if item.strip()]
print(result)

['Hello', 'world', '.', 'This', ',', 'is', 'a', 'test', '.']


We can expand what we split on to create more sophisticated tokenizers.

In [19]:
text = "Hello, world. Is this-- a test?"
result = re.split(r'([,.:;?_!"()\']|--|\s)', text)
result = [item.strip() for item in result if item.strip()]
print(result)

['Hello', ',', 'world', '.', 'Is', 'this', '--', 'a', 'test', '?']


Now we have a tokenizer, we apply it to Ediwth Wharton's entire short story.

In [20]:
preprocessed = re.split(r'([,.:;?_!"()\']|--|\s)', raw_text)
preprocessed = [item.strip() for item in preprocessed if item.strip()]
print(len(preprocessed))

4690


In [21]:
print(preprocessed[:30])

['I', 'HAD', 'always', 'thought', 'Jack', 'Gisburn', 'rather', 'a', 'cheap', 'genius', '--', 'though', 'a', 'good', 'fellow', 'enough', '--', 'so', 'it', 'was', 'no', 'great', 'surprise', 'to', 'me', 'to', 'hear', 'that', ',', 'in']


## Converting tokens into token IDs

We next need to convert these string tokens into integers. To do this, we have to build a vocabulary which defines how we map each unique word and special character to a unique integer. We create a list of all unique tokens from `preprocessed` and sort them alphabetically to determine the vocabulary size.

In [22]:
all_words = sorted(set(preprocessed))
vocab_size = len(all_words)
print(vocab_size)

1130


Let's create the vocabulary and print the first 51 entries for illustration purposes.

In [23]:
vocab = {token:integer for integer, token in enumerate(all_words)}
for i, item in enumerate(vocab.items()):
    print(item)
    if i >= 50:
        break

('!', 0)
('"', 1)
("'", 2)
('(', 3)
(')', 4)
(',', 5)
('--', 6)
('.', 7)
(':', 8)
(';', 9)
('?', 10)
('A', 11)
('Ah', 12)
('Among', 13)
('And', 14)
('Are', 15)
('Arrt', 16)
('As', 17)
('At', 18)
('Be', 19)
('Begin', 20)
('Burlington', 21)
('But', 22)
('By', 23)
('Carlo', 24)
('Chicago', 25)
('Claude', 26)
('Come', 27)
('Croft', 28)
('Destroyed', 29)
('Devonshire', 30)
('Don', 31)
('Dubarry', 32)
('Emperors', 33)
('Florence', 34)
('For', 35)
('Gallery', 36)
('Gideon', 37)
('Gisburn', 38)
('Gisburns', 39)
('Grafton', 40)
('Greek', 41)
('Grindle', 42)
('Grindles', 43)
('HAD', 44)
('Had', 45)
('Hang', 46)
('Has', 47)
('He', 48)
('Her', 49)
('Hermia', 50)


Let's implement a tokenizer class in Python with an `encode` method to split text into tokens and carry out the string-to-integer mapping to produce token IDs via a vocabulary. In addition, let's implement a `decode` method that carries out the inverse mapping for the same vocabulary.

In [24]:
class SimpleTokenizerV1:
    def __init__(self, vocab):
        self.str_to_int = vocab
        self.int_to_str = {i:s for s, i in vocab.items()}
        
    def encode(self, text): # process input text into token IDs
        preprocessed = re.split(r'([,.:;?_!"()\']|--|\s)', text)
        preprocessed = [
            item.strip() for item in preprocessed if item.strip()
        ]
        ids = [self.str_to_int[s] for s in preprocessed]
        return ids
    
    def decode(self, ids): # process token IDs into text
        text = " ".join([self.int_to_str[i] for i in ids])
        text = re.sub(r'\s+([,.?!"()\'])', r'\1', text) # remove spaces before specified punctuation
        return text
    
tokenizer = SimpleTokenizerV1(vocab)

text = """"It's the last he painted, you know," 
        Mrs. Gisburn said with pardonable pride."""
ids = tokenizer.encode(text)
print(ids)

[1, 56, 2, 850, 988, 602, 533, 746, 5, 1126, 596, 5, 1, 67, 7, 38, 851, 1108, 754, 793, 7]


In [25]:
print(tokenizer.decode(ids))

" It' s the last he painted, you know," Mrs. Gisburn said with pardonable pride.


Now let's try to tokenize a new text sample not in our short story.

In [26]:
text = "Hello, do you like tea?"
print(tokenizer.encode(text)) # Anticipating key error on Hello

KeyError: 'Hello'

This highlights the need to consider large and diverse training sets to extend the vocabulary when working on LLMs.

## Adding special context tokens

We need to modify the tokenizer to handle unknown words. We also want to address the usage and addition of special context tokens that can enhance the model's understanding of context and other relevant information from the text. The special tokens can include markers for unknown words and document boundaries, for example. 

Specifically, we will modify the tokenizer to use an `<|unk|>` token if it encounters a word that is not part of the vocabulary. In addition, we'll add a token between unrelated texts. This helps the LLM understand that although these text sources are concatenated for training, they are, in fact, unrelated.

In [27]:
all_tokens = sorted(list(set(preprocessed)))
all_tokens.extend(["<|endoftext|>", "<|unk|>"]) # adding to the list of unique words
vocab = {token:integer for integer, token in enumerate(all_tokens)}
print(len(vocab.items()))

1132


Another quick check to see that indeed our new tokens have been added to the vocabulary.

In [28]:
for i, item in enumerate(list(vocab.items())[-5:]):
    print(item)

('younger', 1127)
('your', 1128)
('yourself', 1129)
('<|endoftext|>', 1130)
('<|unk|>', 1131)


Now we can create our improved tokenizer (could have used inheritance here for concision).

In [29]:
class SimpleTokenizerV2:
    def __init__(self, vocab):
        self.str_to_int = vocab
        self.int_to_str = {i:s for s,i in vocab.items()}
        
    def encode(self, text):
        preprocessed = re.split(r'([,.:;?_!"()\']|--|\s)', text)
        preprocessed = [
            item.strip() for item in preprocessed if item.strip()
        ]
        preprocessed = [item if item in self.str_to_int else "<|unk|>" for item in preprocessed]
        ids = [self.str_to_int[s] for s in preprocessed]
        return ids
    
    def decode(self, ids): 
        text = " ".join([self.int_to_str[i] for i in ids])
        text = re.sub(r'\s+([,.?!"()\'])', r'\1', text) 
        return text

Now we can test on new text.

In [30]:
text1 = "Hello, do you like tea?"
text2 = "In the sunlit terraces of the palace."
text = " <|endoftext|> ".join((text1, text2))
print(text)

Hello, do you like tea? <|endoftext|> In the sunlit terraces of the palace.


In [31]:
tokenizer = SimpleTokenizerV2(vocab)
print(tokenizer.encode(text))

[1131, 5, 355, 1126, 628, 975, 10, 1130, 55, 988, 956, 984, 722, 988, 1131, 7]


In [32]:
print(tokenizer.decode(tokenizer.encode(text)))

<|unk|>, do you like tea? <|endoftext|> In the sunlit terraces of the <|unk|>.


## Byte pair encoding

This is what trained GPT-2 and GPT-3. We implement it here using the `tiktoken` library.

In [33]:
from importlib.metadata import version
import tiktoken
print("tiktoken version: ", version("tiktoken"))

tiktoken version:  0.7.0


In [37]:
tokenizer = tiktoken.get_encoding("gpt2")
text = (
    "Hello, do you like tea? <|endoftext|> In the sunlit terraces "
    "of someunknownPlace."
)
integers = tokenizer.encode(text, allowed_special={"<|endoftext|>"})
print(integers)

[15496, 11, 466, 345, 588, 8887, 30, 220, 50256, 554, 262, 4252, 18250, 8812, 2114, 286, 617, 34680, 27271, 13]


As with our custom tokenizer class, we can also parse them back into text with decoding.

In [38]:
strings = tokenizer.decode(integers)
print(strings)

Hello, do you like tea? <|endoftext|> In the sunlit terraces of someunknownPlace.


### Exercise 2.1 Byte pair encoding for unknown words

Try the BPE tokenizer from the tiktoken library on the unknown words "Akwirw ier" and print the individual token IDs. Then, call the decode function on each of the resulting integers in this list to reproduce the mapping shown in figure 2.11. Lastly, call the decode method on the token IDs to check whether it can reconstruct the original input, "Akwirw ier".

In [39]:
unknown_words = "Akwirw ier"
token_ids = tokenizer.encode(unknown_words)
print(token_ids)

[33901, 86, 343, 86, 220, 959]


In [45]:
mapping = {tokenizer.decode([x]): x for x in token_ids}
print(mapping)

{'Ak': 33901, 'w': 86, 'ir': 343, ' ': 220, 'ier': 959}


In [40]:
decoded_ids = tokenizer.decode(token_ids)
print(decoded_ids)

Akwirw ier


## Data sampling with a sliding window